# SemRel Dataset — Feature Engineering
**COS 760 Group Project**  
Loads cleaned Afrikaans train/dev/test splits and engineers three feature groups:

| Group | Features |
|---|---|
| **Lexical** | Jaccard (word & char), word-count diff, avg word count, longest common subsequence |
| **String**  | Normalised edit distance, character overlap ratio |
| **Semantic**| TF-IDF cosine similarity, multilingual sentence-embedding cosine similarity |

Outputs: `./features/afr_{train,dev,test}_features.csv`

## 1. Install & Import Dependencies

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn sentence-transformers -q


In [2]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from difflib import SequenceMatcher
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

DATA_DIR    = './cleaned_data'
OUTPUT_DIR  = './features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SPLITS = ['train', 'dev', 'test']

print('Setup complete.')


Setup complete.


## 2. Load Cleaned Data

In [3]:
def load_split(lang: str, split: str, data_dir: str = DATA_DIR) -> pd.DataFrame:
    path = os.path.join(data_dir, f'{lang}_{split}.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'File not found: {path}\nRun 02_preprocessing.ipynb first.'
        )
    return pd.read_csv(path)

data = {}
for split in SPLITS:
    try:
        data[split] = load_split('afr', split)
        print(f'Loaded {split}: {len(data[split])} rows  |  '
              f'columns: {data[split].columns.tolist()}')
    except FileNotFoundError as e:
        print(e)

print('\nSample:')
display(data['train'].head(3))


Loaded train: 300 rows  |  columns: ['sentence1', 'sentence2', 'score', 'lang', 'split']
Loaded dev: 75 rows  |  columns: ['sentence1', 'sentence2', 'score', 'lang', 'split']
Loaded test: 375 rows  |  columns: ['sentence1', 'sentence2', 'score', 'lang', 'split']

Sample:


,sentence1,sentence2,score,lang,split
0,Die sand en steen wat n dam moes vorm het net ...,Op hierdie manier kan n paar sinkkuipe langs m...,0.47,afr,train
1,Vir elke dag van die jaar is daar 'n kort aanh...,Begin elke dag met hierdie aanhaling en dra di...,0.62,afr,train
2,"Omdat lig nie om n voorwerp kan beweeg nie, vo...",Baie voorwerpe laat wel n hoeveelheid lig deur...,0.84,afr,train


## 3. Lexical Features

In [4]:
def tokenise(text: str) -> list[str]:
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r'[a-zA-ZÀ-ÿ0-9]+', str(text).lower())


def jaccard_words(s1: str, s2: str) -> float:
    """Jaccard similarity on word sets."""
    t1, t2 = set(tokenise(s1)), set(tokenise(s2))
    if not t1 and not t2:
        return 1.0
    return len(t1 & t2) / len(t1 | t2)


def jaccard_char_ngrams(s1: str, s2: str, n: int = 3) -> float:
    """Jaccard similarity on character n-gram sets."""
    def ngrams(text):
        t = str(text).lower().replace(' ', '')
        return set(t[i:i+n] for i in range(len(t) - n + 1))
    g1, g2 = ngrams(s1), ngrams(s2)
    if not g1 and not g2:
        return 1.0
    return len(g1 & g2) / len(g1 | g2)


def lcs_ratio(s1: str, s2: str) -> float:
    """Longest common subsequence length / max(len(s1), len(s2))."""
    return SequenceMatcher(None, str(s1).lower(), str(s2).lower()).ratio()


def word_count_diff(s1: str, s2: str) -> float:
    """Absolute word count difference."""
    return abs(len(tokenise(s1)) - len(tokenise(s2)))


def avg_word_count(s1: str, s2: str) -> float:
    return (len(tokenise(s1)) + len(tokenise(s2))) / 2


def extract_lexical_features(df: pd.DataFrame) -> pd.DataFrame:
    feats = pd.DataFrame()
    feats['jaccard_word']       = df.apply(lambda r: jaccard_words(r['sentence1'], r['sentence2']), axis=1)
    feats['jaccard_char3']      = df.apply(lambda r: jaccard_char_ngrams(r['sentence1'], r['sentence2'], 3), axis=1)
    feats['jaccard_char2']      = df.apply(lambda r: jaccard_char_ngrams(r['sentence1'], r['sentence2'], 2), axis=1)
    feats['lcs_ratio']          = df.apply(lambda r: lcs_ratio(r['sentence1'], r['sentence2']), axis=1)
    feats['word_count_diff']    = df.apply(lambda r: word_count_diff(r['sentence1'], r['sentence2']), axis=1)
    feats['avg_word_count']     = df.apply(lambda r: avg_word_count(r['sentence1'], r['sentence2']), axis=1)
    return feats

print('Extracting lexical features...')
lex_feats = {split: extract_lexical_features(data[split]) for split in SPLITS if split in data}
print('Done.')
print(lex_feats['train'].describe().round(4))


Extracting lexical features...
Done.
       jaccard_word  jaccard_char3  jaccard_char2  lcs_ratio  word_count_diff  \
count      300.0000       300.0000       300.0000   300.0000         300.0000   
mean         0.2024         0.1454         0.2770     0.4025           2.9667   
std          0.1425         0.1233         0.1297     0.1468           2.2589   
min          0.0000         0.0000         0.0667     0.1341           0.0000   
25%          0.0800         0.0382         0.1760     0.2853           1.0000   
50%          0.1962         0.1228         0.2518     0.3782           2.0000   
75%          0.2872         0.2365         0.3753     0.4957           4.0000   
max          0.6875         0.5789         0.6765     0.8671          12.0000   

       avg_word_count  
count        300.0000  
mean          14.2733  
std            2.2106  
min            6.5000  
25%           13.0000  
50%           14.5000  
75%           16.0000  
max           19.0000  


## 4. String Distance Features

In [5]:
def normalised_edit_distance(s1: str, s2: str) -> float:
    """
    Normalised Levenshtein distance using SequenceMatcher.
    Returns 0 (identical) to 1 (completely different).
    """
    similarity = SequenceMatcher(None, str(s1).lower(), str(s2).lower()).ratio()
    return 1.0 - similarity


def char_overlap_ratio(s1: str, s2: str) -> float:
    """Proportion of characters in s1 that appear in s2 (multiset intersection)."""
    c1, c2 = Counter(str(s1).lower()), Counter(str(s2).lower())
    intersection = sum((c1 & c2).values())
    total = sum(c1.values())
    return intersection / total if total > 0 else 0.0


def extract_string_features(df: pd.DataFrame) -> pd.DataFrame:
    feats = pd.DataFrame()
    feats['edit_distance_norm'] = df.apply(
        lambda r: normalised_edit_distance(r['sentence1'], r['sentence2']), axis=1)
    feats['char_overlap_ratio'] = df.apply(
        lambda r: char_overlap_ratio(r['sentence1'], r['sentence2']), axis=1)
    return feats

print('Extracting string distance features...')
str_feats = {split: extract_string_features(data[split]) for split in SPLITS if split in data}
print('Done.')
print(str_feats['train'].describe().round(4))


Extracting string distance features...
Done.
       edit_distance_norm  char_overlap_ratio
count            300.0000            300.0000
mean               0.5975              0.7626
std                0.1468              0.1287
min                0.1329              0.3187
25%                0.5043              0.6888
50%                0.6218              0.7734
75%                0.7147              0.8609
max                0.8659              1.0000


## 5. TF-IDF Cosine Similarity

In [6]:
def extract_tfidf_features(
    train_df: pd.DataFrame,
    split_dfs: dict[str, pd.DataFrame],
) -> dict[str, pd.DataFrame]:
    """
    Fit TF-IDF on train sentences, then compute cosine similarity
    for each (sentence1, sentence2) pair across all splits.
    """
    # Fit on all train sentences combined
    all_train_sents = (
        train_df['sentence1'].tolist() + train_df['sentence2'].tolist()
    )
    vectorizer = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    vectorizer.fit(all_train_sents)

    result = {}
    for split, df in split_dfs.items():
        vecs1 = vectorizer.transform(df['sentence1'])
        vecs2 = vectorizer.transform(df['sentence2'])
        # Diagonal of the cosine similarity matrix = per-pair similarity
        sims  = np.array(
            [cosine_similarity(vecs1[i], vecs2[i])[0, 0] for i in range(len(df))]
        )
        result[split] = pd.DataFrame({'tfidf_cosine': sims})
    return result

print('Extracting TF-IDF cosine similarity features...')
tfidf_feats = extract_tfidf_features(data['train'], {s: data[s] for s in SPLITS if s in data})
print('Done.')
print(tfidf_feats['train'].describe().round(4))


Extracting TF-IDF cosine similarity features...
Done.
       tfidf_cosine
count      300.0000
mean         0.1527
std          0.1474
min          0.0000
25%          0.0137
50%          0.1247
75%          0.2565
max          0.6662


## 6. Multilingual Sentence Embedding Cosine Similarity
Uses `paraphrase-multilingual-mpnet-base-v2` from `sentence-transformers`,
which supports Afrikaans via its multilingual training.

In [ ]:
EMBEDDING_MODEL = 'paraphrase-multilingual-mpnet-base-v2'

def extract_embedding_features(
    split_dfs: dict[str, pd.DataFrame],
    model_name: str = EMBEDDING_MODEL,
) -> dict[str, pd.DataFrame]:
    """
    Encode sentence pairs with a multilingual sentence transformer
    and return cosine similarity for each pair.
    """
    print(f'Loading model: {model_name}')
    model = SentenceTransformer(model_name)

    result = {}
    for split, df in split_dfs.items():
        print(f'  Encoding {split} ({len(df)} pairs)...', end=' ')
        emb1 = model.encode(df['sentence1'].tolist(), batch_size=64,
                            show_progress_bar=False, convert_to_numpy=True)
        emb2 = model.encode(df['sentence2'].tolist(), batch_size=64,
                            show_progress_bar=False, convert_to_numpy=True)

        # Row-wise cosine similarity
        norm1 = emb1 / (np.linalg.norm(emb1, axis=1, keepdims=True) + 1e-10)
        norm2 = emb2 / (np.linalg.norm(emb2, axis=1, keepdims=True) + 1e-10)
        sims  = (norm1 * norm2).sum(axis=1)

        result[split] = pd.DataFrame({'embedding_cosine': sims})
        print(f'done. mean sim = {sims.mean():.4f}')

    return result

emb_feats = extract_embedding_features({s: data[s] for s in SPLITS if s in data})
print('\nEmbedding features extracted.')
print(emb_feats['train'].describe().round(4))


Loading model: paraphrase-multilingual-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3918.55it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoding train (300 pairs)... done. mean sim = 0.4810
  Encoding dev (75 pairs)... done. mean sim = 0.4866
  Encoding test (375 pairs)... 

## 7. Combine All Features

In [ ]:
def combine_features(split: str) -> pd.DataFrame:
    """Concatenate all feature groups + the gold score for one split."""
    parts = [
        lex_feats[split],
        str_feats[split],
        tfidf_feats[split],
        emb_feats[split],
        data[split][['score']].reset_index(drop=True),
    ]
    return pd.concat(parts, axis=1)

feature_data = {}
for split in SPLITS:
    if split in lex_feats:
        feature_data[split] = combine_features(split)
        print(f'{split}: {feature_data[split].shape}  |  '
              f'columns: {feature_data[split].columns.tolist()}')


## 8. Feature Distribution & Correlation

In [ ]:
# Distribution of each feature in the train split
train_feat_df = feature_data['train']
feat_cols = [c for c in train_feat_df.columns if c != 'score']

n_cols = 3
n_rows = -(-len(feat_cols) // n_cols)   # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(feat_cols):
    axes[i].hist(train_feat_df[col], bins=25, color='steelblue',
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('Value', fontsize=8)
    axes[i].set_ylabel('Count', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Afrikaans — feature distributions (train)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Correlation matrix including score
corr = feature_data['train'].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature correlation matrix (train)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Correlation of each feature with the gold score
score_corr = (
    feature_data['train'].corr()['score']
    .drop('score')
    .sort_values(key=abs, ascending=False)
)

print('Feature correlation with gold score (train):')
print(score_corr.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in score_corr.values]
ax.barh(score_corr.index[::-1], score_corr.values[::-1], color=colors[::-1],
        edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r with score')
ax.set_title('Feature–score correlation (train)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_score_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Save Feature Matrices
Saves `afr_{train,dev,test}_features.csv` to `./features/`.
Each file contains all engineered features **plus** the gold `score` column.

In [ ]:
for split, df in feature_data.items():
    path = f'{OUTPUT_DIR}/afr_{split}_features.csv'
    df.to_csv(path, index=False)
    print(f'Saved: {path}  ({df.shape[0]} rows, {df.shape[1]} columns)')


## 10. Helper Loader for Model Notebooks
Copy this function into any downstream notebook.

In [ ]:
def load_features(split: str, lang: str = 'afr',
                  feature_dir: str = './features') -> pd.DataFrame:
    """
    Load the feature CSV for a given split.
    Returns a DataFrame with all features and the gold 'score' column.

    Example:
        train_feats = load_features('train')
        X_train = train_feats.drop(columns=['score'])
        y_train = train_feats['score']
    """
    path = os.path.join(feature_dir, f'{lang}_{split}_features.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Not found: {path}\nRun 03_feature_engineering.ipynb first.'
        )
    return pd.read_csv(path)


# Quick verification
print('Testing loader...')
for split in SPLITS:
    try:
        df = load_features(split)
        X  = df.drop(columns=['score'])
        y  = df['score']
        print(f'  {split}: X={X.shape}  y={y.shape}')
    except FileNotFoundError as e:
        print(e)


## 11. Final Summary

In [ ]:
print('=' * 50)
print('FEATURE ENGINEERING COMPLETE — FINAL SUMMARY')
print('=' * 50)

for split in SPLITS:
    if split in feature_data:
        df  = feature_data[split]
        X   = df.drop(columns=['score'])
        print(f'  {split:<6}: {len(df)} rows  |  {X.shape[1]} features')

print()
print('Feature columns:')
for col in X.columns:
    print(f'  - {col}')

print()
print('Feature files saved to:', OUTPUT_DIR)
print()
print('Load in your model notebook with:')
print('  from 03_feature_engineering import load_features   # if imported as module')
print('  # or copy the load_features() function defined above')
print('  X_train = load_features("train").drop(columns=["score"])')
print('  y_train = load_features("train")["score"]')
